# Nepal Earthquake Building Damage Prediction

**Algorithm:** Random Forest Classifier  
**Dataset:** DrivenData Richter’s Predictor (2015 Nepal earthquake)  
**Objective:** Predict building damage grade (1: low, 2: moderate, 3: severe).

Run cells **from top to bottom** in Google Colab. Upload `train_values.csv` and `train_labels.csv` when prompted. Files uploaded to Colab are temporary.

## 1. Load dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

if not all(os.path.exists(f) for f in ["train_values.csv", "train_labels.csv"]):
    from google.colab import files
    files.upload()

values = pd.read_csv("train_values.csv")
labels = pd.read_csv("train_labels.csv")
print("Building data:", values.shape)
print("Damage labels:", labels.shape)

## 2. Prepare the dataset

Join tables by `building_id`, one-hot encode categorical features, and keep the **test split separate** during training.

In [ ]:
data = values.merge(labels, on="building_id", validate="one_to_one")
print("Combined dataset:", data.shape)
print("Total missing entries:", data.isna().sum().sum())

X = data.drop(columns=["building_id", "damage_grade"])
y = data["damage_grade"]
X = pd.get_dummies(X, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Encoded features:", len(X.columns))

## 3. Train one Random Forest model

In [ ]:
import time
start = time.time()
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Training time (seconds):", round(time.time() - start, 2))

## 4. Evaluate on the held-out test set

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("Micro F1:", round(f1_score(y_test, y_pred, average="micro"), 4))
print("
Classification report:
", classification_report(y_test, y_pred))
print("Confusion matrix:
", confusion_matrix(y_test, y_pred, labels=[1, 2, 3]))

## 5. Predict the damage grade for a test building

Change `building_number` to examine a different building in the test set. The class probabilities are model estimates, not calibrated guarantees.

In [ ]:
damage_names = {1: "Low Damage", 2: "Moderate Damage", 3: "Severe Damage"}
building_number = 10
building = X_test.iloc[[building_number]]
prediction = int(model.predict(building)[0])
actual = int(y_test.iloc[building_number])
probabilities = model.predict_proba(building)[0]

print("NEPAL EARTHQUAKE DAMAGE PREDICTION")
for col in ["geo_level_1_id", "count_floors_pre_eq", "age", "area_percentage", "height_percentage"]:
    print(col, ":", building.iloc[0][col])
print("Predicted damage:", damage_names[prediction])
print("Actual damage:", damage_names[actual])
for grade, probability in zip(model.classes_, probabilities):
    print(damage_names[int(grade)], ":", round(probability * 100, 2), "%")

## 6. Results visualizations

These charts are also saved as PNG files for the report and presentation.

In [ ]:
names = ["Low Damage", "Moderate Damage", "Severe Damage"]
counts = y.value_counts().reindex([1, 2, 3])
plt.figure(figsize=(8, 5))
sns.barplot(x=names, y=counts.values)
plt.title("Distribution of Earthquake Damage Levels")
plt.xlabel("Damage Level")
plt.ylabel("Number of Buildings")
plt.tight_layout()
plt.savefig("damage_distribution.png", dpi=300)
plt.show()

cm = confusion_matrix(y_test, y_pred, labels=[1, 2, 3])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Damage")
plt.ylabel("Actual Damage")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

importance = pd.Series(model.feature_importances_, index=X_train.columns).nlargest(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=importance.values, y=importance.index)
plt.title("Top 10 Important Features")
plt.xlabel("Feature Importance")
plt.ylabel("Building Features")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=300)
plt.show()

## 7. Save trained model

This creates a model file in the active Colab session. Download it from the Files sidebar if you need to keep it after the session ends.

In [ ]:
joblib.dump({"model": model, "features": X_train.columns.tolist()}, "nepal_earthquake_model.pkl")
print("Saved nepal_earthquake_model.pkl")

## Reference

Yitian Liang, *Application of Machine Learning Methods to Predict the Level of Buildings Damage in the Nepal Earthquake*, Stanford CS229 Final Project Report, 2021.  
Dataset: [DrivenData — Richter’s Predictor: Modeling Earthquake Damage](https://www.drivendata.org/competitions/57/nepal-earthquake/).

The code uses **only Random Forest**, and the evaluation numbers are generated by running this notebook on its own held-out test set.